# Agente de Caminho Mínimo — Mapa da Romênia

Agente de resolução de problemas sobre o mapa clássico da Romênia (AIMA): formulação do espaço de estados, função de custo g(n) e busca da rota de menor custo entre duas cidades.

**Técnica:** Busca de custo uniforme  
**Referência:** Russell & Norvig, *Inteligência Artificial* (AIMA)  
**Contexto:** disciplina de Inteligência Artificial — Análise e Desenvolvimento de Sistemas, FATEC Taubaté

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devcauas/agentes-ia/blob/main/notebooks/03-caminho-minimo-romenia.ipynb)


In [4]:
import heapq

# =============================================================================
# FUNDAMENTAÇÃO TEÓRICA: AGENTES DE RESOLUÇÃO DE PROBLEMAS
# =============================================================================
"""
Um Agente de Resolução de Problemas é um tipo de agente baseado em objetivos que
utiliza representações atômicas do mundo para encontrar sequências de ações que
levam a estados desejados.

1. Formulação do Problema: Processo de definir estados, ações e custos.
2. Espaço de Estados: Conjunto de todos os estados alcançáveis a partir do inicial.
3. Busca: Processo de explorar o espaço de estados através de uma Árvore de Busca.
4. Custo do Caminho: Função g(n) que atribui um valor numérico à trajetória.
"""

# =============================================================================
# REPRESENTAÇÃO DO CONHECIMENTO: O MAPA DA ROMÊNIA
# =============================================================================

# Grafo de adjacência: Cidades e distâncias reais (custos de aresta)
MAPA_ROMENIA = {
    'Arad': [('Zerind', 75), ('Sibiu', 140), ('Timisoara', 118)],
    'Zerind': [('Arad', 75), ('Oradea', 71)],
    'Oradea': [('Zerind', 71), ('Sibiu', 151)],
    'Sibiu': [('Arad', 140), ('Oradea', 151), ('Fagaras', 99), ('Rimnicu Vilcea', 80)],
    'Timisoara': [('Arad', 118), ('Lugoj', 111)],
    'Lugoj': [('Timisoara', 111), ('Mehadia', 70)],
    'Mehadia': [('Lugoj', 70), ('Dobreta', 75)],
    'Dobreta': [('Mehadia', 75), ('Craiova', 120)],
    'Craiova': [('Dobreta', 120), ('Rimnicu Vilcea', 146), ('Pitesti', 138)],
    'Rimnicu Vilcea': [('Sibiu', 80), ('Craiova', 146), ('Pitesti', 97)],
    'Fagaras': [('Sibiu', 99), ('Bucareste', 211)],
    'Pitesti': [('Rimnicu Vilcea', 97), ('Craiova', 138), ('Bucareste', 101)],
    'Bucareste': [('Fagaras', 211), ('Pitesti', 101), ('Giurgiu', 90), ('Urziceni', 85)],
    'Giurgiu': [('Bucareste', 90)],
    'Urziceni': [('Bucareste', 85), ('Vaslui', 142), ('Hirsova', 98)],
    'Hirsova': [('Urziceni', 98), ('Eforie', 86)],
    'Eforie': [('Hirsova', 86)],
    'Vaslui': [('Urziceni', 142), ('Iasi', 92)],
    'Iasi': [('Vaslui', 92), ('Neamt', 87)],
    'Neamt': [('Iasi', 87)]
}

# Heurística h(n): Distância em linha reta até Bucareste (Admissível e Consistente)
HEURISTICA_BUCARESTE = {
    'Arad': 366, 'Bucareste': 0, 'Craiova': 160, 'Dobreta': 242, 'Eforie': 161,
    'Fagaras': 176, 'Giurgiu': 77, 'Hirsova': 151, 'Iasi': 226, 'Lugoj': 244,
    'Mehadia': 241, 'Neamt': 234, 'Oradea': 380, 'Pitesti': 100, 'Rimnicu Vilcea': 193,
    'Sibiu': 253, 'Timisoara': 329, 'Urziceni': 80, 'Vaslui': 199, 'Zerind': 374
}

# =============================================================================
# ESTRUTURA DE DADOS: O NÓ DA BUSCA
# =============================================================================

class No:
    """
    Representa um nó na árvore de busca.
    - estado: Cidade atual.
    - pai: Nó que gerou este nó (para reconstrução de caminho).
    - acao: Estrada tomada para chegar aqui.
    - g_n: Custo real acumulado do início até este nó.
    - h_n: Estimativa do custo deste nó até o objetivo.
    - f_n: Custo total estimado (g + h).
    """
    def __init__(self, estado, pai=None, acao=None, g_n=0, h_n=0):
        self.estado = estado
        self.pai = pai
        self.acao = acao
        self.g_n = g_n
        self.h_n = h_n
        self.f_n = g_n + h_n
        self.profundidade = 0
        if pai:
            self.profundidade = pai.profundidade + 1

    def __lt__(self, outro):
        # Necessário para a fila de prioridade (heapq) em caso de empate no custo
        return self.f_n < outro.f_n

# =============================================================================
# AGENTE DE RESOLUÇÃO DE PROBLEMAS
# =============================================================================

class AgenteCaminhoMinimo:
    def __init__(self, mapa, heuristica):
        self.mapa = mapa
        self.heuristica = heuristica
        self.nos_expandidos = 0

    def reconstruir_caminho(self, no):
        caminho = []
        atual = no
        while atual:
            caminho.append(atual.estado)
            atual = atual.pai
        return caminho[::-1]

    # -------------------------------------------------------------------------
    # PARTE 1: BUSCA DE CUSTO UNIFORME (UCS)
    # -------------------------------------------------------------------------
    """
    CONCEITO: A UCS expande o nó 'n' com o menor custo acumulado g(n).
    - Otimalidade: É ótima se o custo de cada passo for >= ε > 0.
    - Relação: É essencialmente o Algoritmo de Dijkstra para encontrar o caminho mais curto.
    - Estratégia: Busca Cega (não conhece a localização do objetivo).
    """
    def busca_custo_uniforme(self, origem, destino):
        self.nos_expandidos = 0
        no_inicial = No(estado=origem)

        # Fronteira como Fila de Prioridade (Priority Queue)
        fronteira = []
        heapq.heappush(fronteira, (no_inicial.g_n, no_inicial))

        # Conjunto Explorador (Closed Set) para Busca em Grafo
        explorado = set()

        print(f"\nIniciando UCS: {origem} -> {destino}")

        while fronteira:
            custo, no_atual = heapq.heappop(fronteira)

            if no_atual.estado == destino:
                return self.reconstruir_caminho(no_atual), no_atual.g_n

            if no_atual.estado not in explorado:
                explorado.add(no_atual.estado)
                self.nos_expandidos += 1

                # Expansão de sucessores
                for (vizinho, custo_trecho) in self.mapa.get(no_atual.estado, []):
                    custo_total = no_atual.g_n + custo_trecho
                    no_filho = No(estado=vizinho, pai=no_atual, acao=vizinho, g_n=custo_total)

                    if vizinho not in explorado:
                        heapq.heappush(fronteira, (no_filho.g_n, no_filho))

        return None, float('inf')

    # -------------------------------------------------------------------------
    # PARTE 2: BUSCA A* (A-STAR)
    # -------------------------------------------------------------------------
    """
    CONCEITO: A* minimiza f(n) = g(n) + h(n).
    - h(n): Heurística admissível (nunca superestima o custo real).
    - Eficiência: Busca Informada que "poda" ramos irrelevantes do grafo.
    - Otimalidade: Garante o caminho mínimo se h(n) for admissível (em árvores)
      ou consistente (em grafos).
    """
    def busca_a_estrela(self, origem, destino):
        self.nos_expandidos = 0
        no_inicial = No(estado=origem, h_n=self.heuristica[origem])

        fronteira = []
        heapq.heappush(fronteira, (no_inicial.f_n, no_inicial))

        # Dicionário para controle de custo e prevenção de ciclos
        visitados = {origem: no_inicial.g_n}

        print(f"\nIniciando A*: {origem} -> {destino}")

        while fronteira:
            f_prioridade, no_atual = heapq.heappop(fronteira)

            if no_atual.estado == destino:
                return self.reconstruir_caminho(no_atual), no_atual.g_n

            self.nos_expandidos += 1

            for (vizinho, custo_trecho) in self.mapa.get(no_atual.estado, []):
                g_sucessor = no_atual.g_n + custo_trecho

                # Se o vizinho não foi visitado ou encontramos um caminho mais barato
                if vizinho not in visitados or g_sucessor < visitados[vizinho]:
                    visitados[vizinho] = g_sucessor
                    h_sucessor = self.heuristica[vizinho]
                    no_filho = No(vizinho, no_atual, vizinho, g_sucessor, h_sucessor)
                    heapq.heappush(fronteira, (no_filho.f_n, no_filho))

        return None, float('inf')

# =============================================================================
# INTERFACE PRINCIPAL E EXECUÇÃO
# =============================================================================

def executar_projeto():
    agente = AgenteCaminhoMinimo(MAPA_ROMENIA, HEURISTICA_BUCARESTE)
    origem = "Hirsova"
    destino = "Timisoara"

    print("-" * 50)
    print("PROJETO IA: AGENTE DE CAMINHO MÍNIMO (ROMÊNIA)")
    print("-" * 50)

    # Execução UCS
    caminho_ucs, custo_ucs = agente.busca_custo_uniforme(origem, destino)
    nos_ucs = agente.nos_expandidos

    # Execução A*
    caminho_astar, custo_astar = agente.busca_a_estrela(origem, destino)
    nos_astar = agente.nos_expandidos

    # Exibição de Resultados
    print("\n" + "="*50)
    print(f"RESULTADOS: {origem} para {destino}")
    print("="*50)

    print(f"\n[BUSCA DE CUSTO UNIFORME - UCS]")
    print(f"Caminho: {' -> '.join(caminho_ucs)}")
    print(f"Custo Total: {custo_ucs} km")
    print(f"Nós Expandidos: {nos_ucs}")

    print(f"\n[BUSCA A* - BUSCA INFORMADA]")
    print(f"Caminho: {' -> '.join(caminho_astar)}")
    print(f"Custo Total: {custo_astar} km")
    print(f"Nós Expandidos: {nos_astar}")

    print("\n" + "-"*50)
    print("ANÁLISE COMPARATIVA")
    print("-"*50)
    print(f"1. Ambos os algoritmos encontraram o custo ótimo de {custo_astar} km.")
    print(f"2. O A* expandiu {nos_ucs - nos_astar} nós a menos que a UCS.")
    print(f"3. A eficiência do A* deve-se à heurística h(n) que guia a busca.")
    print(f"4. A UCS é uma busca cega, explorando círculos concêntricos de custo.")

if __name__ == "__main__":
    executar_projeto()

--------------------------------------------------
PROJETO IA: AGENTE DE CAMINHO MÍNIMO (ROMÊNIA)
--------------------------------------------------

Iniciando UCS: Hirsova -> Timisoara

Iniciando A*: Hirsova -> Timisoara

RESULTADOS: Hirsova para Timisoara

[BUSCA DE CUSTO UNIFORME - UCS]
Caminho: Hirsova -> Urziceni -> Bucareste -> Pitesti -> Rimnicu Vilcea -> Sibiu -> Arad -> Timisoara
Custo Total: 719 km
Nós Expandidos: 19

[BUSCA A* - BUSCA INFORMADA]
Caminho: Hirsova -> Urziceni -> Bucareste -> Pitesti -> Rimnicu Vilcea -> Sibiu -> Arad -> Timisoara
Custo Total: 719 km
Nós Expandidos: 19

--------------------------------------------------
ANÁLISE COMPARATIVA
--------------------------------------------------
1. Ambos os algoritmos encontraram o custo ótimo de 719 km.
2. O A* expandiu 0 nós a menos que a UCS.
3. A eficiência do A* deve-se à heurística h(n) que guia a busca.
4. A UCS é uma busca cega, explorando círculos concêntricos de custo.
